In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier
from sklearn.metrics import accuracy_score , precision_score , recall_score,f1_score
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import dagshub
dagshub.init(repo_owner='Aayush10671', repo_name='emotion-detection', mlflow=True)

mlflow.set_tracking_uri('https://dagshub.com/Aayush10671/emotion-detection.mlflow')


Accessing as Aayush10671

Initialized MLflow to track repo "Aayush10671/emotion-detection"

Repository Aayush10671/emotion-detection initialized!

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv')
df.head(5)

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


In [4]:
# data preprocessing



def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)


def normalize_text(df):
    """Normalize the text data."""
    try:
        # Use 'content' instead of 'text' since that's the actual column name
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        raise
  

In [5]:
df = normalize_text(df)
df.head(5)

,tweet_id,sentiment,content
0,1956967341,empty,tiffanylue know listenin bad habit earlier sta...
1,1956967666,sadness,layin n bed headache ughhhh waitin call
2,1956967696,sadness,funeral ceremony gloomy friday
3,1956967789,enthusiasm,want hang friend soon
4,1956968416,neutral,dannycastillo want trade someone houston ticke...


In [6]:
X = df['sentiment'].isin(['happiness' , 'sadness'])
df = df[X]

In [7]:
df['sentiment'] = df['sentiment'].replace({'happiness':1 , 'sadness':0})

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_6732\3399834364.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'happiness':1 , 'sadness':0})


In [8]:
df.sample(5)

,tweet_id,sentiment,content
35762,1753176667,1,watching snl ucsd girl laughing twitter blackb...
34513,1752895240,1,totally got hit one bar guest ha ha yay me
15786,1964983071,0,aquafreak too depresses thinking it
25579,1695136784,1,goooooood morning
22326,1694402032,1,taylorswift omg taylor london near scotland pl...


In [9]:
mlflow.set_experiment("Bow vs tfidf")

vectorizer = {
    'BoW' : CountVectorizer(),
    'TF-IDF': TfidfVectorizer()
}

algo = {
    "LogisticRegression" : LogisticRegression(),
    "MultinomialNB" : MultinomialNB(),
    "XGboost" : XGBClassifier(),
    "RandomForestClassifier" : RandomForestClassifier(),
    "GradientBoostingClassifier" : GradientBoostingClassifier()
}

2026/07/17 11:32:07 INFO mlflow.tracking.fluent: Experiment with name 'Bow vs tfidf' does not exist. Creating a new experiment.


In [11]:
with mlflow.start_run(run_name = "All experiments") as parent_run:
    for algo_name , algos in algo.items():
        for vec_name,vectorizers in vectorizer.items():
            with mlflow.start_run(run_name = f"{algo_name} with {vec_name}",nested = True) as child_run:
                X = vectorizers.fit_transform(df['content'])
                y = df['sentiment']
                X_train , X_test , y_train, y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)
                mlflow.log_param("vectorizer" , vec_name)
                mlflow.log_param("algorithm" , algo_name)
                mlflow.log_param("test_size" , 0.2)
                model = algos
                model.fit(X_train , y_train)

                if algo_name == "LogisticRegression":
                    mlflow.log_param("C" , model.C)
                elif algo_name == "MultinomialNB":
                    mlflow.log_param("alpha" , model.alpha)
                elif algo_name == "XGboost":
                    mlflow.log_param("n_estimators" , model.n_estimators)
                    mlflow.log_param("learning_rate" , model.learning_rate)
                elif algo_name == "RandomForestClassifier":
                    mlflow.log_param("n_estimators" , model.n_estimators)
                    mlflow.log_param("max_depth" , model.max_depth)
                elif algo_name == "GradientBoostingClassifier":
                    mlflow.log_param("n_estimators" , model.n_estimators)
                    mlflow.log_param("learning_rate" , model.learning_rate)
                    mlflow.log_param("max_depth" , model.max_depth)
                
                y_pred = model.predict(X_test)
                accuracy = accuracy_score(y_pred , y_test)
                precision = precision_score(y_pred , y_test)
                recall = recall_score(y_pred , y_test)
                f1 = f1_score(y_pred , y_test)

                mlflow.log_metric("accuracy" , accuracy)
                mlflow.log_metric("precision" , precision)
                mlflow.log_metric("recall" , recall)
                mlflow.log_metric("f1-score" , f1)
                mlflow.sklearn.log_model(model , "model")
                import os
                notebook_path = 'exp2.ipynb'
                os.system(f"jupyter nbconvert --to notebook --execute --inplace  {notebook_path}")
                mlflow.log_artifact(notebook_path)

                print(f"algorithm {algo_name} and vectorzer = {vec_name}")
                print("accuracy = " , accuracy)
                print("recall = " , recall)
                print("precision = ", precision)
                print("f1-score = ", f1)



2026/07/17 11:49:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/17 11:49:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


algorithm LogisticRegression and vectorzer = BoW
accuracy =  0.7937349397590362
recall =  0.7846750727449079
precision =  0.7970443349753694
f1-score =  0.7908113391984359
🏃 View run LogisticRegression with BoW at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1/runs/3fea4a05ef1044fd95978dabe3f9023c
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1


2026/07/17 11:49:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/17 11:49:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


algorithm LogisticRegression and vectorzer = TF-IDF
accuracy =  0.7942168674698795
recall =  0.777882797731569
precision =  0.8108374384236453
f1-score =  0.79401833092137
🏃 View run LogisticRegression with TF-IDF at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1/runs/03f77dc50d154256b1bc923d29f64537
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1


2026/07/17 11:50:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/17 11:50:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

algorithm RandomForestClassifier and vectorzer = TF-IDF
accuracy =  0.7677108433734939
recall =  0.7711088504577823
precision =  0.7467980295566502
f1-score =  0.7587587587587588
🏃 View run RandomForestClassifier with TF-IDF at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1/runs/cfc6adec86124277905e6682eda3be06
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1


2026/07/17 11:57:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/17 11:57:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


algorithm GradientBoostingClassifier and vectorzer = BoW
accuracy =  0.7272289156626506
recall =  0.798140770252324
precision =  0.5921182266009852
f1-score =  0.6798642533936652
🏃 View run GradientBoostingClassifier with BoW at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1/runs/fcb85f62d2464a5b81e7acd3203b8fab
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1


2026/07/17 11:58:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/17 11:58:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


algorithm GradientBoostingClassifier and vectorzer = TF-IDF
accuracy =  0.7248192771084337
recall =  0.8057851239669421
precision =  0.5763546798029556
f1-score =  0.672027570361861
🏃 View run GradientBoostingClassifier with TF-IDF at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1/runs/b2ea649d218b483b9b41a337576dc1ab
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1
🏃 View run All experiments at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1/runs/37e7a28a3247489bba568f4c09b00a02
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/1
